
# Roxy notebook example: Order- and disorder-promoting residue content descriptors

This notebook is a **reference implementation example** for the **order/disorder-promoting residue descriptor family** in Roxy.

These descriptors summarize whether a sequence is enriched in residues that are commonly associated with:

- intrinsic disorder
- structural order / compact folding
- local flexibility
- local rigidity
- compositional bias toward disordered or ordered sequence regimes

They do **not** use structure prediction or external models. Everything is inferred directly from residue content and simple sequence-derived profiles.

## Covered outputs

This notebook implements examples such as:

- fraction of disorder-promoting residues
- fraction of order-promoting residues
- disorder/order balance
- disorder/order ratio
- local disorder-promoting patch burden
- local order-promoting patch burden
- alternating order/disorder transitions
- longest disorder-promoting run
- longest order-promoting run
- N-terminal and C-terminal disorder/order asymmetry
- flexibility-associated proxy content
- proline/glycine-rich disorder-support proxy
- aromatic/aliphatic order-support proxy
- class-style implementation for later migration into Roxy


In [1]:

from itertools import groupby

import numpy as np
import pandas as pd


## Demo dataset

In [2]:

df_demo = pd.DataFrame(
    {
        "sequence_id": [
            "orddis_1",
            "orddis_2",
            "orddis_3",
            "orddis_4",
            "orddis_5",
            "orddis_6",
        ],
        "sequence": [
            "MKWVTFISLLFLFSSAYSRGVFRR",
            "GGGGGGGGGGGGGGG",
            "KRRKRRKRRKRRDDDDEE",
            "ACDEFGHIKLMNPQRSTVWY",
            "PPPPGSSSSSTTTTNNQQQ",
            "MSTNPKPQRITLKDGNKVELV",
        ],
        "label": ["A", "B", "A", "B", "A", "B"],
    }
)

df_demo


,sequence_id,sequence,label
0,orddis_1,MKWVTFISLLFLFSSAYSRGVFRR,A
1,orddis_2,GGGGGGGGGGGGGGG,B
2,orddis_3,KRRKRRKRRKRRDDDDEE,A
3,orddis_4,ACDEFGHIKLMNPQRSTVWY,B
4,orddis_5,PPPPGSSSSSTTTTNNQQQ,A
5,orddis_6,MSTNPKPQRITLKDGNKVELV,B


## Constants

In [3]:

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

DISORDER_PROMOTING = set("ARGQSEPK")
ORDER_PROMOTING = set("CWYFILNV")

FLEXIBILITY_RELATED = set("GSP")
RIGIDITY_RELATED = set("ILVFYW")
PRO_GLY_RICH = set("PG")
AROMATIC_ALIPHATIC_ORDER = set("FWYILV")
POLAR_DISORDER_SUPPORT = set("STNQDEKR")


## Helper functions

In [4]:

def clean_sequence(seq: str) -> str:
    if pd.isna(seq):
        return ""
    seq = str(seq).strip().upper().replace("*", "")
    return "".join([aa for aa in seq if aa in STANDARD_AA])


def fraction_from_group(seq: str, aa_group) -> float:
    if len(seq) == 0:
        return np.nan
    return sum(aa in aa_group for aa in seq) / len(seq)


def count_from_group(seq: str, aa_group) -> int:
    return sum(aa in aa_group for aa in seq)


def safe_ratio(a: float, b: float) -> float:
    if b == 0:
        return np.nan
    return a / b


def windows(seq: str, size: int):
    if len(seq) < size:
        return []
    return [seq[i:i+size] for i in range(len(seq) - size + 1)]


def fraction_above_threshold(values, threshold: float) -> float:
    if len(values) == 0:
        return np.nan
    return float(np.mean(np.array(values) > threshold))


def longest_run(seq: str, aa_group) -> int:
    binary = [1 if aa in aa_group else 0 for aa in seq]
    runs = [len(list(group)) for value, group in groupby(binary) if value == 1]
    return max(runs) if runs else 0


def transition_fraction(seq: str, group_a, group_b) -> float:
    if len(seq) < 2:
        return np.nan
    total = len(seq) - 1
    hits = 0
    for i in range(total):
        a = seq[i]
        b = seq[i + 1]
        if (a in group_a and b in group_b) or (a in group_b and b in group_a):
            hits += 1
    return hits / total


def terminal_segment(seq: str, side: str = "N", window: int = 10) -> str:
    if side == "N":
        return seq[:window]
    if side == "C":
        return seq[-window:]
    raise ValueError("side must be 'N' or 'C'")


## Core descriptor function

In [5]:

def order_disorder_descriptors(seq: str, window_sizes=(5, 7), terminal_window=10) -> dict:
    seq = clean_sequence(seq)

    out = {
        "orddis_length": len(seq),
        "orddis_valid_residue_count": len(seq),
    }

    if len(seq) == 0:
        return out

    disorder_frac = fraction_from_group(seq, DISORDER_PROMOTING)
    order_frac = fraction_from_group(seq, ORDER_PROMOTING)

    out["orddis_disorder_fraction"] = disorder_frac
    out["orddis_order_fraction"] = order_frac
    out["orddis_disorder_order_balance"] = disorder_frac - order_frac
    out["orddis_order_disorder_balance"] = order_frac - disorder_frac
    out["orddis_disorder_order_ratio"] = safe_ratio(disorder_frac, order_frac)
    out["orddis_order_disorder_ratio"] = safe_ratio(order_frac, disorder_frac)

    out["orddis_flexibility_fraction"] = fraction_from_group(seq, FLEXIBILITY_RELATED)
    out["orddis_rigidity_fraction"] = fraction_from_group(seq, RIGIDITY_RELATED)
    out["orddis_pro_gly_fraction"] = fraction_from_group(seq, PRO_GLY_RICH)
    out["orddis_aromatic_aliphatic_order_fraction"] = fraction_from_group(seq, AROMATIC_ALIPHATIC_ORDER)
    out["orddis_polar_disorder_support_fraction"] = fraction_from_group(seq, POLAR_DISORDER_SUPPORT)

    out["orddis_longest_disorder_run"] = longest_run(seq, DISORDER_PROMOTING)
    out["orddis_longest_order_run"] = longest_run(seq, ORDER_PROMOTING)
    out["orddis_order_disorder_transition_fraction"] = transition_fraction(seq, ORDER_PROMOTING, DISORDER_PROMOTING)

    # Terminal asymmetry
    nterm = terminal_segment(seq, side="N", window=terminal_window)
    cterm = terminal_segment(seq, side="C", window=terminal_window)

    out["orddis_nterm_disorder_fraction"] = fraction_from_group(nterm, DISORDER_PROMOTING)
    out["orddis_cterm_disorder_fraction"] = fraction_from_group(cterm, DISORDER_PROMOTING)
    out["orddis_nterm_order_fraction"] = fraction_from_group(nterm, ORDER_PROMOTING)
    out["orddis_cterm_order_fraction"] = fraction_from_group(cterm, ORDER_PROMOTING)

    out["orddis_terminal_disorder_asymmetry"] = out["orddis_nterm_disorder_fraction"] - out["orddis_cterm_disorder_fraction"]
    out["orddis_terminal_order_asymmetry"] = out["orddis_nterm_order_fraction"] - out["orddis_cterm_order_fraction"]

    # Local patches
    for window_size in window_sizes:
        ws = windows(seq, window_size)

        disorder_profile = [fraction_from_group(w, DISORDER_PROMOTING) for w in ws]
        order_profile = [fraction_from_group(w, ORDER_PROMOTING) for w in ws]

        out[f"orddis_w{window_size}_disorder_patch_fraction"] = fraction_above_threshold(disorder_profile, threshold=0.5)
        out[f"orddis_w{window_size}_order_patch_fraction"] = fraction_above_threshold(order_profile, threshold=0.5)
        out[f"orddis_w{window_size}_disorder_mean"] = float(np.mean(disorder_profile)) if len(disorder_profile) > 0 else np.nan
        out[f"orddis_w{window_size}_order_mean"] = float(np.mean(order_profile)) if len(order_profile) > 0 else np.nan
        out[f"orddis_w{window_size}_disorder_amplitude"] = float(np.max(disorder_profile) - np.min(disorder_profile)) if len(disorder_profile) > 0 else np.nan
        out[f"orddis_w{window_size}_order_amplitude"] = float(np.max(order_profile) - np.min(order_profile)) if len(order_profile) > 0 else np.nan

    return out


## Functional usage on one sequence

In [6]:

example = order_disorder_descriptors(df_demo.loc[0, "sequence"], window_sizes=(5, 7), terminal_window=10)
list(example.items())[:20]


[('orddis_length', 24),
 ('orddis_valid_residue_count', 24),
 ('orddis_disorder_fraction', 0.4166666666666667),
 ('orddis_order_fraction', 0.5),
 ('orddis_disorder_order_balance', -0.08333333333333331),
 ('orddis_order_disorder_balance', 0.08333333333333331),
 ('orddis_disorder_order_ratio', 0.8333333333333334),
 ('orddis_order_disorder_ratio', 1.2),
 ('orddis_flexibility_fraction', 0.20833333333333334),
 ('orddis_rigidity_fraction', 0.5),
 ('orddis_pro_gly_fraction', 0.041666666666666664),
 ('orddis_aromatic_aliphatic_order_fraction', 0.5),
 ('orddis_polar_disorder_support_fraction', 0.375),
 ('orddis_longest_disorder_run', 3),
 ('orddis_longest_order_run', 5),
 ('orddis_order_disorder_transition_fraction', 0.34782608695652173),
 ('orddis_nterm_disorder_fraction', 0.2),
 ('orddis_cterm_disorder_fraction', 0.7),
 ('orddis_nterm_order_fraction', 0.6),
 ('orddis_cterm_order_fraction', 0.3)]

## Apply descriptors to the full dataset

In [7]:

df_orddis = pd.concat(
    [
        df_demo,
        df_demo["sequence"].apply(
            lambda x: order_disorder_descriptors(
                x,
                window_sizes=(5, 7),
                terminal_window=10,
            )
        ).apply(pd.Series),
    ],
    axis=1,
)

df_orddis.head()


,sequence_id,sequence,label,orddis_length,orddis_valid_residue_count,orddis_disorder_fraction,orddis_order_fraction,orddis_disorder_order_balance,orddis_order_disorder_balance,orddis_disorder_order_ratio,...,orddis_w5_disorder_mean,orddis_w5_order_mean,orddis_w5_disorder_amplitude,orddis_w5_order_amplitude,orddis_w7_disorder_patch_fraction,orddis_w7_order_patch_fraction,orddis_w7_disorder_mean,orddis_w7_order_mean,orddis_w7_disorder_amplitude,orddis_w7_order_amplitude
0,orddis_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24.0,24.0,0.416667,0.500000,-0.083333,0.083333,0.833333,...,0.400000,0.54,0.8,0.8,0.388889,0.611111,0.404762,0.547619,0.714286,0.714286
1,orddis_2,GGGGGGGGGGGGGGG,B,15.0,15.0,1.000000,0.000000,1.000000,-1.000000,NaN,...,1.000000,0.00,0.0,0.0,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2,orddis_3,KRRKRRKRRKRRDDDDEE,A,18.0,18.0,0.777778,0.000000,0.777778,-0.777778,NaN,...,0.757143,0.00,0.8,0.0,0.750000,0.000000,0.785714,0.000000,0.571429,0.000000
3,orddis_4,ACDEFGHIKLMNPQRSTVWY,B,20.0,20.0,0.400000,0.400000,0.000000,0.000000,1.000000,...,0.437500,0.35,0.6,0.6,0.357143,0.000000,0.438776,0.346939,0.285714,0.285714
4,orddis_5,PPPPGSSSSSTTTTNNQQQ,A,19.0,19.0,0.684211,0.105263,0.578947,-0.578947,6.500000,...,0.613333,0.12,1.0,0.4,0.538462,0.000000,0.604396,0.098901,0.857143,0.285714


## Inspect descriptor columns

In [8]:

orddis_cols = [c for c in df_orddis.columns if c.startswith("orddis_") and c not in {"orddis_length", "orddis_valid_residue_count"}]
len(orddis_cols), orddis_cols[:18]


(32,
 ['orddis_disorder_fraction',
  'orddis_order_fraction',
  'orddis_disorder_order_balance',
  'orddis_order_disorder_balance',
  'orddis_disorder_order_ratio',
  'orddis_order_disorder_ratio',
  'orddis_flexibility_fraction',
  'orddis_rigidity_fraction',
  'orddis_pro_gly_fraction',
  'orddis_aromatic_aliphatic_order_fraction',
  'orddis_polar_disorder_support_fraction',
  'orddis_longest_disorder_run',
  'orddis_longest_order_run',
  'orddis_order_disorder_transition_fraction',
  'orddis_nterm_disorder_fraction',
  'orddis_cterm_disorder_fraction',
  'orddis_nterm_order_fraction',
  'orddis_cterm_order_fraction'])

In [9]:

df_orddis[
    [
        "sequence_id",
        "orddis_disorder_fraction",
        "orddis_order_fraction",
        "orddis_disorder_order_balance",
        "orddis_longest_disorder_run",
        "orddis_longest_order_run",
        "orddis_w5_disorder_patch_fraction",
        "orddis_terminal_disorder_asymmetry",
    ]
]


,sequence_id,orddis_disorder_fraction,orddis_order_fraction,orddis_disorder_order_balance,orddis_longest_disorder_run,orddis_longest_order_run,orddis_w5_disorder_patch_fraction,orddis_terminal_disorder_asymmetry
0,orddis_1,0.416667,0.500000,-0.083333,3.0,5.0,0.450000,-0.5
1,orddis_2,1.000000,0.000000,1.000000,15.0,0.0,1.000000,0.0
2,orddis_3,0.777778,0.000000,0.777778,12.0,0.0,0.714286,0.4
3,orddis_4,0.400000,0.400000,0.000000,4.0,3.0,0.250000,0.0
4,orddis_5,0.684211,0.105263,0.578947,10.0,2.0,0.600000,0.6
5,orddis_6,0.476190,0.333333,0.142857,5.0,2.0,0.470588,0.2


## Dataset-level summary

In [10]:

orddis_summary = (
    df_orddis[orddis_cols]
    .mean(axis=0, numeric_only=True)
    .sort_values(ascending=False)
    .rename("mean_value")
    .reset_index()
    .rename(columns={"index": "descriptor"})
)

orddis_summary.head(15)


,descriptor,mean_value
0,orddis_longest_disorder_run,8.166667
1,orddis_disorder_order_ratio,2.440476
2,orddis_longest_order_run,2.000000
3,orddis_nterm_disorder_fraction,0.700000
4,orddis_w5_disorder_amplitude,0.666667
5,orddis_w7_disorder_mean,0.627830
6,orddis_disorder_fraction,0.625808
7,orddis_w5_disorder_mean,0.622898
8,orddis_w7_disorder_patch_fraction,0.594638
9,orddis_cterm_disorder_fraction,0.583333


## Sanity checks

In [11]:

assert "orddis_disorder_fraction" in df_orddis.columns
assert "orddis_order_fraction" in df_orddis.columns
assert "orddis_disorder_order_ratio" in df_orddis.columns
assert "orddis_longest_disorder_run" in df_orddis.columns
assert "orddis_w5_disorder_patch_fraction" in df_orddis.columns
assert "orddis_terminal_disorder_asymmetry" in df_orddis.columns
assert df_orddis["orddis_length"].min() > 0

print(f"Number of order/disorder descriptor columns: {len(orddis_cols)}")
print("Order/disorder descriptor checks passed.")


Number of order/disorder descriptor columns: 32
Order/disorder descriptor checks passed.


## Class-style implementation closer to the real package

In [12]:

class OrderDisorderDescriptors:
    """Example class-style implementation for later migration into Roxy."""

    def __init__(self, window_sizes=(5, 7), terminal_window=10):
        self.window_sizes = tuple(window_sizes)
        self.terminal_window = int(terminal_window)

    def transform_sequence(self, seq: str) -> dict:
        return order_disorder_descriptors(
            seq,
            window_sizes=self.window_sizes,
            terminal_window=self.terminal_window,
        )

    def transform(self, sequences) -> pd.DataFrame:
        return pd.DataFrame([self.transform_sequence(seq) for seq in sequences])


orddis_transformer = OrderDisorderDescriptors(window_sizes=(5, 7), terminal_window=10)
orddis_matrix = orddis_transformer.transform(df_demo["sequence"].tolist())
orddis_matrix.head()


,orddis_length,orddis_valid_residue_count,orddis_disorder_fraction,orddis_order_fraction,orddis_disorder_order_balance,orddis_order_disorder_balance,orddis_disorder_order_ratio,orddis_order_disorder_ratio,orddis_flexibility_fraction,orddis_rigidity_fraction,...,orddis_w5_disorder_mean,orddis_w5_order_mean,orddis_w5_disorder_amplitude,orddis_w5_order_amplitude,orddis_w7_disorder_patch_fraction,orddis_w7_order_patch_fraction,orddis_w7_disorder_mean,orddis_w7_order_mean,orddis_w7_disorder_amplitude,orddis_w7_order_amplitude
0,24,24,0.416667,0.500000,-0.083333,0.083333,0.833333,1.200000,0.208333,0.5,...,0.400000,0.54,0.8,0.8,0.388889,0.611111,0.404762,0.547619,0.714286,0.714286
1,15,15,1.000000,0.000000,1.000000,-1.000000,NaN,0.000000,1.000000,0.0,...,1.000000,0.00,0.0,0.0,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2,18,18,0.777778,0.000000,0.777778,-0.777778,NaN,0.000000,0.000000,0.0,...,0.757143,0.00,0.8,0.0,0.750000,0.000000,0.785714,0.000000,0.571429,0.000000
3,20,20,0.400000,0.400000,0.000000,0.000000,1.000000,1.000000,0.150000,0.3,...,0.437500,0.35,0.6,0.6,0.357143,0.000000,0.438776,0.346939,0.285714,0.285714
4,19,19,0.684211,0.105263,0.578947,-0.578947,6.500000,0.153846,0.526316,0.0,...,0.613333,0.12,1.0,0.4,0.538462,0.000000,0.604396,0.098901,0.857143,0.285714


## Merge transformer output back to the dataset

In [13]:

df_orddis_class = pd.concat([df_demo, orddis_matrix], axis=1)
df_orddis_class.head()


,sequence_id,sequence,label,orddis_length,orddis_valid_residue_count,orddis_disorder_fraction,orddis_order_fraction,orddis_disorder_order_balance,orddis_order_disorder_balance,orddis_disorder_order_ratio,...,orddis_w5_disorder_mean,orddis_w5_order_mean,orddis_w5_disorder_amplitude,orddis_w5_order_amplitude,orddis_w7_disorder_patch_fraction,orddis_w7_order_patch_fraction,orddis_w7_disorder_mean,orddis_w7_order_mean,orddis_w7_disorder_amplitude,orddis_w7_order_amplitude
0,orddis_1,MKWVTFISLLFLFSSAYSRGVFRR,A,24,24,0.416667,0.500000,-0.083333,0.083333,0.833333,...,0.400000,0.54,0.8,0.8,0.388889,0.611111,0.404762,0.547619,0.714286,0.714286
1,orddis_2,GGGGGGGGGGGGGGG,B,15,15,1.000000,0.000000,1.000000,-1.000000,NaN,...,1.000000,0.00,0.0,0.0,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000
2,orddis_3,KRRKRRKRRKRRDDDDEE,A,18,18,0.777778,0.000000,0.777778,-0.777778,NaN,...,0.757143,0.00,0.8,0.0,0.750000,0.000000,0.785714,0.000000,0.571429,0.000000
3,orddis_4,ACDEFGHIKLMNPQRSTVWY,B,20,20,0.400000,0.400000,0.000000,0.000000,1.000000,...,0.437500,0.35,0.6,0.6,0.357143,0.000000,0.438776,0.346939,0.285714,0.285714
4,orddis_5,PPPPGSSSSSTTTTNNQQQ,A,19,19,0.684211,0.105263,0.578947,-0.578947,6.500000,...,0.613333,0.12,1.0,0.4,0.538462,0.000000,0.604396,0.098901,0.857143,0.285714



## Suggested next refactor into the package

A clean migration path into Roxy would be:

- move residue sets into `roxy/core/constants.py`
- move helper logic into `roxy/sequence/order_disorder.py`
- expose a class such as `OrderDisorderDescriptors`
- allow configurable:
  - residue sets for order/disorder
  - window sizes
  - terminal window size
  - selected local/global summaries
- add tests for:
  - empty sequences
  - strongly disorder-promoting sequences
  - strongly order-promoting sequences
  - mixed sequences with terminal asymmetry
  - lower-case input
  - invalid characters removed during cleaning


## Optional export

In [14]:
# df_orddis.to_csv("demo_order_disorder_descriptors.csv", index=False)
